# 02b — EfficientNet-B0 with 4 unfrozen blocks

In run 02, EfficientNet-B0 only fine-tuned its last 2 blocks (28% of its weights) at lr 1e-5 and was still under-fitting (93.7% val acc) while ResNet-18 (94% of weights fine-tuned) reached 99.0%.
Here we unfreeze the last **4** blocks (**92%** of weights — comparable to ResNet) and fine-tune with **lr 1e-4**.

**Kaggle:** Add Input → *140k Real and Fake Faces* **and** Add Input → Your Work → output of the *02 training* notebook; Accelerator = GPU; Internet = On; then *Save Version → Save & Run All*.

The output of this notebook contains the new `efficientnet_best.pth` **and** `resnet_best.pth`, so notebooks 03/04 only need this notebook's output attached.

Theory: [`docs/04_training.md`](../docs/04_training.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE or ON_COLAB:
    PROJECT = "/kaggle/working/project" if ON_KAGGLE else "/content/project"
    if os.path.exists(PROJECT):
        !git -C {PROJECT} pull -q
    else:
        !git clone -q {REPO} {PROJECT}
    %cd {PROJECT}
    EXTRA = "" if ON_KAGGLE else "kagglehub"
    !pip install -q grad-cam {EXTRA}
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset: Kaggle -> attached input is found automatically; Colab/local -> downloaded once (~4 GB) ---
from src import config
if ON_KAGGLE:
    for ckpt in config.import_kaggle_checkpoints():      # notebooks 03/04: models from notebook 02's output
        print("trained model:", ckpt)
if not (config.DATA_DIR / "train").exists():
    if ON_KAGGLE:
        raise RuntimeError("Dataset not found - attach '140k Real and Fake Faces' with Add Input")
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Keep the run-02 EfficientNet for comparison
The setup cell copied run 02's checkpoints into `checkpoints/`. Save the old EfficientNet under a different name before it is overwritten (only `*_best.pth` files are picked up by evaluation and the app).

In [ ]:
!cp checkpoints/efficientnet_best.pth checkpoints/efficientnet_2blocks.pth
!ls -lh checkpoints

## Train — 3 frozen epochs, then 7 epochs with the last 4 blocks unfrozen at lr 1e-4

In [ ]:
!python -m src.train --model efficientnet_b0 --checkpoint-dir checkpoints --num-workers 4 --unfreeze-blocks 4 --lr-finetune 1e-4

## Training curves

In [ ]:
from IPython.display import Image as Show
Show('outputs/plots/training_curves_efficientnet.png')

## Package for download

In [ ]:
!zip -qr results_02b.zip outputs checkpoints/efficientnet_best.pth checkpoints/resnet_best.pth checkpoints/efficientnet_2blocks.pth
!ls -lh results_02b.zip checkpoints

## Observations

_Compare with run 02 (2 blocks, lr 1e-5): 93.71% val acc, val loss 0.158._